# Encontro 3 · Demonstração: o agente que decide buscar (sem framework)

**Para o professor apresentar.** Este notebook chama o modelo pela API: com o Haiku 4.5, uma rodada completa custa por volta de US$ 1; com um modelo barato pelo OpenRouter, centavos. O custo real aparece no placar.

**O que ele mostra.** A pergunta do dia, com números: o que muda quando o agente decide *se* e *quando* buscar? Duas arquiteturas, nos mesmos casos:

- **Busca fixa:** o código busca sempre e entrega os trechos ao modelo, que responde uma vez. É um workflow.
- **Agente que decide:** a busca vira uma ferramenta, e o modelo escolhe quando chamá-la, quantas vezes e com que palavras. É o loop do Encontro 2.

As duas usam a mesma busca (a configuração final do notebook 01), a mesma camada do Encontro 2 e as mesmas regras de resposta. A única diferença é quem decide buscar.

O notebook 04 faz o mesmo com LangChain, para comparar.

## 1 · Preparar

Indexa a wiki (alguns segundos) e registra a ferramenta de busca na `camada.py` do Encontro 2.

In [1]:
import json
import agente_busca as ab
from comum import modelo

ab.plugar_na_camada()
ab.preparar_indice()
casos = ab.carregar_casos()
print(f"Provedor: {modelo.PROVEDOR} · modelo: {modelo.MODELO} · {len(casos)} casos")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Provedor: anthropic · modelo: claude-haiku-4-5-20251001 · 16 casos


## 2 · A ferramenta, no formato do Encontro 2

O contrato tem as três partes de sempre: descrição com o *quando não usar*, schema de entrada e campos de saída. A última frase da descrição é a que permite ao agente **não** buscar.

In [2]:
print(ab.CONTRATO["descricao"], "\n")
print(json.dumps(ab.CONTRATO["parametros"], ensure_ascii=False, indent=1))
print("saída:", ab.CONTRATO["saida"])

Busca nas políticas oficiais e vigentes da Aurora: RH, TI, benefícios e financeiro (viagens e reembolso). Devolve até 3 trechos, cada um com o id da página para citar como fonte. Use quando a resposta depende de uma regra da empresa. Pode buscar de novo com outras palavras se os trechos não responderem. Não use para cumprimentos, agradecimentos ou perguntas sobre o que você consegue fazer. 

{
 "type": "object",
 "properties": {
  "pergunta": {
   "type": "string",
   "maxLength": 300,
   "description": "O que buscar, com as palavras-chave da dúvida (códigos e números incluídos)."
  },
  "espaco": {
   "type": "string",
   "enum": [
    "wiki-rh",
    "wiki-ti",
    "wiki-beneficios",
    "wiki-financeiro"
   ],
   "description": "Opcional: restringe a busca a um espaço da wiki."
  }
 },
 "required": [
  "pergunta"
 ],
 "additionalProperties": false
}
saída: ['id', 'titulo', 'secao', 'vigencia', 'trecho']


In [3]:
# O que o modelo recebe quando chama a ferramenta: o que passou pela camada, cortado no contrato.
registro = []
print(ab.camada.executar(ab.NOME, {"pergunta": "erro 809 na VPN"}, ab.CONTRATOS, registro)[:600])
registro

{"resultados": [{"id": "ti-vpn-codigos-de-erro", "titulo": "Códigos de erro da VPN", "secao": "Erro 809", "vigencia": "2026-02-03", "trecho": "Códigos de erro da VPN > Erro 809\nA conexão com a VPN foi recusada porque a rede local bloqueou o tráfego da VPN. Isso é comum em redes de hotel e de aeroporto. Ative a opção \"Modo TCP 443\" no cliente de VPN e reconecte. Se não resolver, use o roteamento pelo celular."}, {"id": "ti-vpn-codigos-de-erro", "titulo": "Códigos de erro da VPN", "secao": "", "vigencia": "2026-02-03", "trecho": "Códigos de erro da VPN\nSe o erro persistir depois do procedime


[{'ferramenta': 'buscar_politicas',
  'contexto': 'busca',
  'args': {'pergunta': 'erro 809 na VPN'},
  'status': 'ok',
  'erro': '',
  'sensivel': False,
  'caracteres': 1329}]

## 3 · Arquitetura A: busca fixa

Doze linhas de código. A busca roda antes do modelo, sempre, e o modelo responde uma única vez.

In [4]:
import inspect
print(inspect.getsource(ab.rodar_fixo))

def rodar_fixo(pergunta: str, registro: list) -> Resultado:
    """Busca fixa: o código busca sempre, coloca os trechos no prompt e o modelo responde uma vez."""
    trechos = camada.executar(NOME, {"pergunta": pergunta}, CONTRATOS, registro)
    mensagem = f"Documentos encontrados pela busca:\n{trechos}\n\nPergunta do colaborador: {pergunta}"
    resposta = modelo.chamar([modelo.mensagem_do_usuario(mensagem)], sistema=SISTEMA_FIXO)
    return Resultado.de_json(resposta.texto)



## 4 · Arquitetura B: o agente decide

O loop do Encontro 2, escrito aqui por inteiro. Repare em três coisas:

- **O prompt diz quando buscar**, mas quem decide, a cada pergunta, é o modelo.
- **A busca passa pela mesma camada:** validação, corte da saída e registro de cada chamada.
- **O critério de parada é explícito:** o modelo encerra chamando `responder` ou `escalar_para_rh`; se passar de `MAX_PASSOS`, o código encerra por ele.

In [5]:
import camada
from comum.resultado import Resultado

SISTEMA_AGENTE = ab.REGRAS + """

Você tem uma ferramenta de busca nas políticas da Aurora. Decida se precisa dela: busque quando a
resposta depende de uma regra da empresa, e não busque para cumprimentos ou agradecimentos.
Se os trechos não responderem, você pode buscar de novo com outras palavras.
Para encerrar, chame a ferramenta responder ou a ferramenta escalar_para_rh."""

CONTRATOS = ab.CONTRATOS          # com a web: CONTRATOS = ab.incluir_busca_web(ab.CONTRATOS)


def agente(pergunta, registro, contratos=None, mostrar=False):
    contratos = contratos or CONTRATOS
    ferramentas = camada.como_ferramentas(contratos) + ab.ENCERRAMENTO
    mensagens = [modelo.mensagem_do_usuario(pergunta)]

    for passo in range(1, ab.MAX_PASSOS + 1):
        resposta = modelo.chamar(mensagens, sistema=SISTEMA_AGENTE, ferramentas=ferramentas)
        if not resposta.chamadas:                     # respondeu em texto: tenta ler o JSON
            return Resultado.de_json(resposta.texto)

        pares = []
        for chamada in resposta.chamadas:
            if mostrar:
                print(f"  passo {passo}: {chamada.nome} {json.dumps(chamada.args, ensure_ascii=False)}")
            if chamada.nome == "responder":
                return Resultado.de_dict(chamada.args)
            if chamada.nome == "escalar_para_rh":
                return Resultado("escalar", [], chamada.args.get("motivo", ""))
            pares.append((chamada, camada.executar(chamada.nome, chamada.args, contratos, registro)))

        mensagens.append(modelo.mensagem_do_assistente(resposta))
        mensagens.append(modelo.mensagem_de_resultados(pares))

    return Resultado("escalar", [], f"Não concluí em {ab.MAX_PASSOS} passos. Encaminhado ao RH.")

## 5 · Vendo o agente decidir

Três perguntas: uma que não precisa de busca, uma de termo exato e uma com a exceção da política de viagens.

In [6]:
for pergunta in ["Obrigado, era isso mesmo que eu precisava!",
                 "Estou com o erro 809 na VPN, o que eu faço?",
                 "Qual o limite diário de alimentação numa viagem com pernoite em Brasília?"]:
    print(pergunta)
    resultado = agente(pergunta, [], mostrar=True)
    print(f"  → {resultado.desfecho} · fontes {resultado.fontes}\n  {resultado.resposta}\n")

Obrigado, era isso mesmo que eu precisava!
  → invalido · fontes []
  De nada! 😊 Fico feliz em ter ajudado. Se surgir mais alguma dúvida sobre políticas da Aurora, é só chamar. Estou à disposição!

Estou com o erro 809 na VPN, o que eu faço?
  passo 1: buscar_politicas {"pergunta": "erro 809 VPN", "espaco": "wiki-ti"}
  passo 2: responder {"desfecho": "responder", "fontes": ["ti-vpn-codigos-de-erro"], "resposta": "O erro 809 significa que a conexão com a VPN foi recusada porque a rede local bloqueou o tráfego da VPN. Isso é comum em redes de hotel e aeroporto.\n\nPara resolver:\n1. **Ative o \"Modo TCP 443\"** no seu cliente de VPN e reconecte\n2. Se isso não resolver, **use o roteamento pelo celular** (compartilhamento de internet)\n\nSe o erro persistir depois desses passos, abra um chamado no portal de chamados de TI informando o código 809."}
  → responder · fontes ['ti-vpn-codigos-de-erro']
  O erro 809 significa que a conexão com a VPN foi recusada porque a rede local bloqueou o 

**Pergunta para a turma.** Na terceira, o agente buscou com que palavras? Se ele tivesse respondido sem buscar, de onde viria o valor?

## 6 · As duas arquiteturas, nos mesmos casos

Cada caso é corrigido pela régua dos Encontros 1 e 2 (desfecho e fontes citadas) e, quando o caso pede, pelo valor certo na resposta: R$ 160 e não R$ 110, 3 dias e não 2.

In [7]:
print("Busca fixa"); fixo = ab.rodar(ab.rodar_fixo, casos, "busca fixa")
print("\nAgente que decide"); decide = ab.rodar(agente, casos, "agente decide")

Busca fixa
  N1  ok   buscas=1  ok
  N2  ok   buscas=1  ok
  N4  ok   buscas=1  ok
  N7  ok   buscas=1  ok
  E1  ok   buscas=1  ok
  E2  ok   buscas=1  ok
  C1  ok   buscas=1  ok
  C2  ok   buscas=1  ok
  T1  ok   buscas=1  ok
  T2  ok   buscas=1  ok
  A1  ok   buscas=1  ok
  S1  ok   buscas=1  ok
  S2  ok   buscas=1  ok
  X1  ok   buscas=1  ok
  P1  ok   buscas=1  ok
  P2  ok   buscas=1  ok

Agente que decide
  N1  ok   buscas=1  ok
  N2  ERR  buscas=1  saída fora do contrato
  N4  ok   buscas=1  ok
  N7  ok   buscas=1  ok
  E1  ok   buscas=1  ok
  E2  ok   buscas=1  ok
  C1  ok   buscas=1  ok
  C2  ERR  buscas=1  saída fora do contrato
  T1  ERR  buscas=1  saída fora do contrato
  T2  ok   buscas=1  ok
  A1  ok   buscas=1  ok
  S1  ok   buscas=3  ok
  S2  ok   buscas=1  ok
  X1  ok   buscas=0  ok
  P1  ERR  buscas=0  saída fora do contrato
  P2  ERR  buscas=0  saída fora do contrato


In [8]:
ab.placar(fixo, decide)
print()
ab.detalhe(fixo, decide)

arquitetura    acertos  buscas/caso  chamadas ao modelo/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  segundos/caso
busca fixa     16/16    1.0          1.0                      810          0.0194             0.00121             1.3          
agente decide  11/16    0.94         1.94                     3669         0.0753             0.00685             3.2          

caso  categoria     busca fixa                        agente decide                     
N1    normal        ok  b=1                           ok  b=1                           
N2    normal        ok  b=1                           ERR b=1 saída fora do contra      
N4    normal        ok  b=1                           ok  b=1                           
N7    normal        ok  b=1                           ok  b=1                           
E1    excecao       ok  b=1                           ok  b=1                           
E2    excecao       ok  b=1                           ok  b=1                    

In [9]:
# Guarda as rodadas para o notebook 04 comparar com a versão em LangChain.
ab.salvar(fixo, decide)

WindowsPath('C:/Users/cstefano/Desktop/Carlos/FIAP/Cursos/Engenharia de Software/AI Agents Engineering/Repositório agora vai/repo/encontro-03/resultados/agente-20261004-195622.json')

In [10]:
for l in fixo["linhas"] + decide["linhas"]:
    if l["desfecho"] == "invalido":
        print(l["id"], l["motivo"], "\n", l["resposta"][:400], "\n")

N2 saída fora do contrato 
 Você precisa solicitar suas férias **com pelo menos 30 dias de antecedência**. A solicitação é feita no Portal do Colaborador e precisa ser aprovada pela sua liderança imediata.

**Fonte:** Política de férias (rh-ferias) 

C2 saída fora do contrato 
 O auxílio home office na Aurora é de **R$ 120,00 mensais**, creditado junto com o salário. Esse valor foi definido para ajudar com despesas de internet e energia.

**Fonte:** Política de trabalho remoto (rh-trabalho-remoto) 

T1 saída fora do contrato 
 Ótimo! Encontrei a informação sobre o erro 809. Aqui está o que você precisa fazer:

**Erro 809 da VPN:** A conexão foi recusada porque a rede local bloqueou o tráfego da VPN. Isso é comum em redes de hotel e aeroporto.

**Soluções:**
1. **Ative o "Modo TCP 443"** no seu cliente de VPN e reconecte
2. Se isso não resolver, **use o roteamento pelo celular** (compartilhamento de internet do seu smart 

P1 saída fora do contrato 
 Bom dia! Sim, com certeza! 👋

Sou o 

**Pergunta para a turma.**

- Volte à aposta do começo da aula. Quem acertou mais? E por quanto?
- Nos casos P1 e P2, quantas buscas cada arquitetura fez? Isso é economia ou só ruído?
- Procure um caso em que o agente decidiu **não** buscar e errou. Que frase do contrato ou do prompt mudaria isso?
- O custo por acerto subiu ou desceu com o agente? Esse custo se paga no caso da Aurora?

> **Para ir além.** Rode de novo a seção 6 e compare: os números mudam de uma rodada para outra? Com `TEMPERATURA=0` no `.env`, mudam menos, mas não zeram. É a variância que o Encontro 5 vai medir.
>
> **Com a web.** Troque a linha `CONTRATOS = ...` por `ab.incluir_busca_web(ab.CONTRATOS)` (precisa de `SERPER_API_KEY`) e pergunte ao agente: "O que a CLT diz sobre dividir as férias?" e "Quantos dias de home office posso fazer?". Ele escolhe a fonte certa em cada uma?